# TeCo averaging: saved-result plots only

**Run all**, using Zyion when mounting Drive. No GPU required. This notebook reads only the completed `teco_average40` results. No training, feature preparation, package installation or new Drive outputs.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 5. Plots from the saved results
These cells only read saved outputs. They do not rerun classification or save extra files to Drive.


In [ ]:
from pathlib import Path
import json, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
OUT=Path('/content/drive/MyDrive/Thesis/Results/word_sentiment_eeg/teco_average40')
status=json.loads((OUT/'complete.json').read_text())
if status['status']!='completed':
    raise RuntimeError('The classification run is incomplete. Check skipped.json before interpreting plots.')
for name,sha in status['checksums'].items():
    path=(OUT/name).resolve()
    if not path.is_relative_to(OUT.resolve()):raise ValueError('Invalid result path')
    if hashlib.sha256(path.read_bytes()).hexdigest()!=sha:raise ValueError('Saved result checksum mismatch: '+name)
scores=pd.read_csv(OUT/'scores.csv')
print('Verified completed TeCo task1 results. Plotting only; no fitting, feature loading or Drive writes.')


### Main comparison
Chance balanced accuracy is 50%. No uncertainty interval or significance test is shown.


In [ ]:
fig,axes=plt.subplots(1,3,figsize=(14,4),sharey=True)
for ax,split in zip(axes,['sentence','word','reader']):
    for model,color in [('logistic','#148f77'),('dummy','#888888')]:
        part=scores[(scores.split==split)&scores.feature.eq('post')&scores.model.eq(model)].sort_values('maximum_trials')
        ax.plot(part.maximum_trials,100*part.balanced_accuracy,'o-',color=color,label=model)
        for x,y in zip(part.maximum_trials,100*part.balanced_accuracy):
            ax.annotate(f'{y:.1f}%',(x,y),xytext=(0,8 if model=='logistic' else -16),textcoords='offset points',ha='center',fontsize=9)
    ax.axhline(50,color='black',ls=':',lw=1)
    ax.set(title='Unseen '+split+'s',xlabel='Maximum trials per same-class average',xticks=[1,40])
axes[0].set_ylabel('Balanced accuracy (%)')
axes[-1].legend()
fig.suptitle('TeCo task1: individual trials versus up to 40-trial averages\nPost-onset EEG, 0–800 ms regional features; descriptive scores')
fig.tight_layout();display(fig);plt.close(fig)


### EEG and control features


In [ ]:
fig,axes=plt.subplots(1,3,figsize=(14,4),sharey=True)
settings=[('post','logistic','Post EEG'),('pre','logistic','Pre EEG'),('nuisance','logistic','Word/eye'),('post','dummy','Dummy')]
for ax,split in zip(axes,['sentence','word','reader']):
    values=[]
    for feature,model,_ in settings:
        row=scores[scores.split.eq(split)&scores.maximum_trials.eq(40)&scores.feature.eq(feature)&scores.model.eq(model)]
        if len(row)!=1:raise ValueError('Missing/duplicate control case')
        values.append(100*float(row.balanced_accuracy.iloc[0]))
    bars=ax.bar([v[2] for v in settings],values,color=['#148f77','#2874a6','#b47a28','#888888'])
    ax.bar_label(bars,fmt='%.1f%%',padding=3,fontsize=9)
    ax.axhline(50,color='black',ls=':',lw=1);ax.set(title='Unseen '+split+'s',ylim=(0,100))
axes[0].set_ylabel('Balanced accuracy (%)')
fig.suptitle('TeCo task1: controls with averages of up to 40\nWord/eye uses word and fixation properties, without EEG')
fig.tight_layout();display(fig);plt.close(fig)


### Which class did the model predict?


In [ ]:
fig,axes=plt.subplots(3,2,figsize=(9,11),layout='constrained')
for r,split in enumerate(['sentence','word','reader']):
    for c,maximum in enumerate([1,40]):
        saved=json.loads((OUT/f'cases/{split}__max{maximum}__post__logistic/result.json').read_text())
        cm=np.asarray(saved['confusion'],dtype=int)
        if cm.shape!=(2,2) or (cm.sum(axis=1)==0).any():raise ValueError('Invalid saved confusion matrix')
        rates=cm/cm.sum(axis=1,keepdims=True)
        ax=axes[r,c];ax.imshow(rates,vmin=0,vmax=1,cmap='Blues')
        for i in range(2):
            for j in range(2):
                ax.text(j,i,f'{rates[i,j]:.1%}\n(n={cm[i,j]})',ha='center',va='center',color='white' if rates[i,j]>.6 else 'black')
        title='Individual trials' if maximum==1 else 'Averages up to 40'
        ax.set(title=f'Unseen {split}s: {title}',xlabel='Predicted class',ylabel='True class',xticks=[0,1],yticks=[0,1],xticklabels=['Negative','Positive'],yticklabels=['Negative','Positive'])
fig.suptitle('TeCo task1: logistic regression confusion matrices\nRows normalized within true class; n counts test examples, not original trials')
display(fig);plt.close(fig)


### Actual group sizes and test-example counts


In [ ]:
main=scores[scores.feature.eq('post')&scores.model.eq('logistic')]
display(main[['split','maximum_trials','test_examples','original_test_trials','actual_mean_trials','actual_min_trials','full40_test_examples']])
print('A requested maximum of 40 can include smaller remainder groups. full40_test_examples counts exact 40-trial test averages.')
print('The same test events appear in the individual and averaged conditions; only their grouping changes.')
